In [42]:
import cv2
import pandas as pd 
import numpy as np

In [43]:
csv_path = "../Data/real_moon_images/PCAM_bounding_boxes.csv"
df = pd.read_csv(csv_path)

In [44]:
print("CSV columns:",df.columns)

CSV columns: Index(['Frame', 'TopLeftCornerX', 'TopLeftCornerY', 'Length', 'Height',
       'Color'],
      dtype='str')


In [45]:
img_name = "PCAM1.png"
img_path = f'../Data/real_moon_images/{img_name}'
raw_img = cv2.imread(img_path,cv2.IMREAD_UNCHANGED)

In [46]:
normalized = cv2.normalize(raw_img,None, alpha = 0, beta = 255, norm_type =cv2.NORM_MINMAX)
img_8bit = np.uint8(normalized)

In [47]:
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
gray_img = cv2.cvtColor(img_8bit, cv2.COLOR_BGRA2GRAY) if img_8bit.ndim == 3 and img_8bit.shape[2] == 4 else img_8bit
gray_img = cv2.cvtColor(img_8bit, cv2.COLOR_BGR2GRAY) if img_8bit.ndim == 3 and img_8bit.shape[2] == 3 else (
    cv2.cvtColor(img_8bit, cv2.COLOR_BGRA2GRAY) if img_8bit.ndim == 3 and img_8bit.shape[2] == 4 else img_8bit
)
visible_img = clahe.apply(gray_img)
visible_bgr = cv2.cvtColor(visible_img,cv2.COLOR_GRAY2BGR)

In [48]:
boxes = df[df['image'] == img_name] if 'image' in df.columns else df

In [49]:
for _, row in boxes.iterrows():
    x = int(row["TopLeftCornerX"])
    y = int(row["TopLeftCornerY"])
    w = int(row["Length"])
    h = int(row["Height"])

    color_name = str(row["Color"]).lower()
    color_map = {
        "green": (0, 255, 0),
        "blue": (255, 0, 0),
        "red": (0, 0, 255),
        "yellow": (0, 255, 255),
    }
    color = color_map.get(color_name, (0, 255, 0))

    cv2.rectangle(visible_bgr, (x, y), (x + w, y + h), color, 2)

In [50]:
output_path ="PCAM2_visualized.png"
cv2.imwrite(output_path,visible_bgr)
print(f"done , check '{output_path} in folder.")

done , check 'PCAM2_visualized.png in folder.
